# Your first LLM API call, done properly

**The Agent Build Log · Build #003 · GenAI & LLM Foundations 3/8**

[Build #001](../01-tokens-and-embeddings/) was what a model reads, [build #002](../02-sampling-parameters/) how it picks
words. This build is about **your code**: making one LLM call that survives production. The 3-line version
everyone copies works in a demo. Real apps also hit rate limits, overloaded servers, timeouts and cut-off answers.

1. [The 3-line call](#1-the-3-line-call)
2. [Read the whole response](#2-read-the-whole-response): `stop_reason`, usage, cost
3. [Streaming: same answer, sooner](#3-streaming-same-answer-sooner)
4. [Real errors, and which ones to retry](#4-real-errors-and-which-ones-to-retry)
5. [Retries under failure](#5-retries-under-failure) (failures injected on purpose)
6. [Timeouts: streaming vs not](#6-timeouts-streaming-vs-not)
7. [The production version](#7-the-production-version)
8. [Findings](#8-findings)

**Needs:** an Anthropic API key in a `.env` file (`ANTHROPIC_API_KEY=...`). Uses **Claude Haiku 4.5**;
the whole notebook costs a few cents. Errors are not billed.

In [1]:
import json, os, random, statistics, time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic
import httpx2

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 70)

MODEL = "claude-haiku-4-5"
PRICE = {"input": 1.00, "output": 5.00}    # $ per 1M tokens, Claude Haiku 4.5
client = anthropic.Anthropic()
print("anthropic SDK", anthropic.__version__, "| model:", MODEL)

anthropic SDK 1.11.0 | model: claude-haiku-4-5


## 1. The 3-line call

This is what most tutorials show. It works, and it hides almost everything you need in production.

In [2]:
t0 = time.perf_counter()
r = client.messages.create(model=MODEL, max_tokens=300,
                           messages=[{"role": "user", "content": "In one sentence, what is an API rate limit?"}])
naive_seconds = time.perf_counter() - t0
print(r.content[0].text)
print(f"\n{naive_seconds:.2f}s")

An API rate limit is a restriction on how many requests a client can make to an API within a specified time period.

0.69s


## 2. Read the whole response

Before trusting `content[0].text`, check **why the model stopped** and **what it cost**. A `200 OK` with
`stop_reason == "max_tokens"` means the answer was **cut off**, and nothing raises an error.

In [3]:
def cost(usage) -> float:
    return (usage.input_tokens * PRICE["input"] + usage.output_tokens * PRICE["output"]) / 1e6

def text_of(msg) -> str:
    return "".join(b.text for b in msg.content if b.type == "text")

prompt = "List 5 reasons an API call can fail, one short line each."
full = client.messages.create(model=MODEL, max_tokens=400, messages=[{"role": "user", "content": prompt}])
cut = client.messages.create(model=MODEL, max_tokens=25, messages=[{"role": "user", "content": prompt}])

anatomy = pd.DataFrame([
    {"max_tokens": 400, "stop_reason": full.stop_reason, "input tokens": full.usage.input_tokens,
     "output tokens": full.usage.output_tokens, "cost $": round(cost(full.usage), 6), "request id": full._request_id[:12] + "…"},
    {"max_tokens": 25, "stop_reason": cut.stop_reason, "input tokens": cut.usage.input_tokens,
     "output tokens": cut.usage.output_tokens, "cost $": round(cost(cut.usage), 6), "request id": cut._request_id[:12] + "…"},
])
print("CUT-OFF ANSWER (HTTP 200, no exception):", repr(text_of(cut)))
anatomy

CUT-OFF ANSWER (HTTP 200, no exception): '# 5 Reasons an API Call Can Fail\n\n1. **Invalid authentication** - Missing or expired credentials/'


,max_tokens,stop_reason,input tokens,output tokens,cost $,request id
0,400,end_turn,23,94,0.000493,req_011CfhpB…
1,25,max_tokens,23,25,0.000148,req_011CfhpB…


## 3. Streaming: same answer, sooner

Streaming doesn't make the model faster. It shows the **first words** while the rest is still being generated.
We time 5 runs of each on the same prompt: **time to first token (TTFT)** and **total time**.

In [4]:
STREAM_PROMPT = "Explain to a junior developer what an API rate limit is and how to handle one. About 150 words."
N = 5

def run_blocking():
    t0 = time.perf_counter()
    m = client.messages.create(model=MODEL, max_tokens=600, messages=[{"role": "user", "content": STREAM_PROMPT}])
    return {"mode": "non-streaming", "first text (s)": time.perf_counter() - t0,
            "total (s)": time.perf_counter() - t0, "output tokens": m.usage.output_tokens}

def run_streaming():
    t0 = time.perf_counter(); first = None
    with client.messages.stream(model=MODEL, max_tokens=600,
                                messages=[{"role": "user", "content": STREAM_PROMPT}]) as stream:
        for _ in stream.text_stream:
            if first is None:
                first = time.perf_counter() - t0
        m = stream.get_final_message()
    return {"mode": "streaming", "first text (s)": first,
            "total (s)": time.perf_counter() - t0, "output tokens": m.usage.output_tokens}

runs = []
for _ in range(N):            # interleave so both modes see the same conditions
    runs.append(run_blocking()); runs.append(run_streaming())
stream_df = pd.DataFrame(runs)
stream_summary = stream_df.groupby("mode")[["first text (s)", "total (s)", "output tokens"]].median().round(2)
stream_summary

,first text (s),total (s),output tokens
mode,,,
non-streaming,3.03,3.03,258.0
streaming,0.60,3.05,258.0


## 4. Real errors, and which ones to retry

These are **real** responses from the API (errors are not billed). The SDK raises a typed exception for each.
We count how many HTTP attempts the SDK made, with its default `max_retries=2`, using a counter on the transport.

In [5]:
class CountingTransport(httpx2.HTTPTransport):
    """Passes requests through and counts them, so we can see the SDK's retries."""
    def __init__(self, **kw):
        super().__init__(**kw); self.calls = 0
    def handle_request(self, request):
        self.calls += 1
        return super().handle_request(request)

def attempt(label, make_client, **kw):
    tr = CountingTransport()
    c = make_client(tr)
    t0 = time.perf_counter()
    try:
        c.messages.create(model=kw.pop("model", MODEL), max_tokens=kw.pop("max_tokens", 20),
                          messages=kw.pop("messages", [{"role": "user", "content": "hi"}]), **kw)
        outcome, status = "ok", 200
    except anthropic.APIStatusError as e:
        outcome, status = type(e).__name__, e.status_code
    except anthropic.APITimeoutError as e:
        outcome, status = type(e).__name__, "-"
    except anthropic.APIConnectionError as e:
        outcome, status = type(e).__name__, "-"
    return {"case": label, "exception": outcome, "status": status,
            "HTTP attempts": tr.calls, "seconds": round(time.perf_counter() - t0, 2)}

default = lambda tr: anthropic.Anthropic(http_client=anthropic.DefaultHttpxClient(transport=tr))
errors = pd.DataFrame([
    attempt("wrong model name", default, model="claude-does-not-exist"),
    attempt("invalid request (empty messages list)", default, messages=[]),
    attempt("bad API key", lambda tr: anthropic.Anthropic(api_key="sk-ant-invalid",
                                                       http_client=anthropic.DefaultHttpxClient(transport=tr))),
    attempt("server unreachable", lambda tr: anthropic.Anthropic(base_url="http://127.0.0.1:9",
                                                              http_client=anthropic.DefaultHttpxClient(transport=tr))),
])
errors

,case,exception,status,HTTP attempts,seconds
0,wrong model name,NotFoundError,404,1,0.23
1,invalid request (empty messages list),BadRequestError,400,1,0.21
2,bad API key,AuthenticationError,401,1,0.14
3,server unreachable,APIConnectionError,-,3,1.42


## 5. Retries under failure

Rate limits (429) and overload (529) can't be triggered on demand, so here we **inject them on purpose**:
a transport that answers a fraction of requests with a real-looking 429 or 529 before they reach the API.
The **retrying is the SDK's own**. We send 40 small requests at a 30% injected failure rate and compare
`max_retries` settings.

In [6]:
class FlakyTransport(CountingTransport):
    """Answers `rate` of requests with an injected 429 or 529; passes the rest to the real API."""
    def __init__(self, rate, seed):
        super().__init__(); self.rate = rate; self.rng = random.Random(seed); self.injected = 0
    def handle_request(self, request):
        if self.rng.random() < self.rate:
            self.calls += 1; self.injected += 1
            status, kind = self.rng.choice([(429, "rate_limit_error"), (529, "overloaded_error")])
            return httpx2.Response(status, request=request,
                                   json={"type": "error", "error": {"type": kind, "message": "injected for this experiment"}})
        return super().handle_request(request)

RATE, REQUESTS, SEED = 0.30, 40, 7

def retry_run(max_retries):
    tr = FlakyTransport(RATE, SEED)
    c = anthropic.Anthropic(max_retries=max_retries, http_client=anthropic.DefaultHttpxClient(transport=tr))
    ok, t0 = 0, time.perf_counter()
    for _ in range(REQUESTS):
        try:
            c.messages.create(model=MODEL, max_tokens=5, messages=[{"role": "user", "content": "Say OK"}]); ok += 1
        except (anthropic.RateLimitError, anthropic.InternalServerError, anthropic.APIStatusError):
            pass
    return {"max_retries": max_retries, "succeeded": f"{ok}/{REQUESTS}", "success %": round(ok / REQUESTS * 100),
            "HTTP attempts": tr.calls, "injected failures": tr.injected,
            "seconds": round(time.perf_counter() - t0, 1)}

retries = pd.DataFrame([retry_run(n) for n in (0, 2, 4)])
retries

,max_retries,succeeded,success %,HTTP attempts,injected failures,seconds
0,0,23/40,57,40,17,13.7
1,2,38/40,95,60,22,31.1
2,4,40/40,100,62,22,41.6


## 6. Timeouts: streaming vs not

The SDK's default timeout is 10 minutes, which is far too long for a user-facing app. But a short timeout on a
long answer fails. We ask for a ~600-word answer with a **2-second timeout** and no retries, both ways.
For streaming, the timeout applies **between chunks**, not to the whole answer.

In [7]:
LONG = "Write about 600 words on how to design retry logic for a web API client."
def timed(fn):
    t0 = time.perf_counter()
    try:
        out = fn(); return {"result": "ok", "output tokens": out, "seconds": round(time.perf_counter() - t0, 2)}
    except anthropic.APITimeoutError:
        return {"result": "APITimeoutError", "output tokens": 0, "seconds": round(time.perf_counter() - t0, 2)}

short = client.with_options(timeout=2.0, max_retries=0)
def blocking():
    return short.messages.create(model=MODEL, max_tokens=1500, messages=[{"role": "user", "content": LONG}]).usage.output_tokens
def streaming():
    with short.messages.stream(model=MODEL, max_tokens=1500, messages=[{"role": "user", "content": LONG}]) as s:
        for _ in s.text_stream:
            pass
        return s.get_final_message().usage.output_tokens

timeouts = pd.DataFrame([{"mode": "non-streaming, 2s timeout", **timed(blocking)},
                         {"mode": "streaming, 2s timeout", **timed(streaming)}])
timeouts

,mode,result,output tokens,seconds
0,"non-streaming, 2s timeout",APITimeoutError,0,2.04
1,"streaming, 2s timeout",ok,1055,12.23


## 7. The production version

Everything above in one small function: a sane timeout, the SDK's retries, streaming, a `stop_reason` check,
usage and cost logged, and errors split into "retryable, already retried" and "fix your code".

In [8]:
prod = anthropic.Anthropic(timeout=anthropic.Timeout(60.0, connect=5.0), max_retries=3)

def ask(prompt: str, max_tokens: int = 1024, on_text=None) -> dict:
    t0 = time.perf_counter(); first = None
    try:
        with prod.messages.stream(model=MODEL, max_tokens=max_tokens,
                                  messages=[{"role": "user", "content": prompt}]) as stream:
            for chunk in stream.text_stream:
                first = first or time.perf_counter() - t0
                if on_text: on_text(chunk)
            msg = stream.get_final_message()
    except (anthropic.RateLimitError, anthropic.InternalServerError, anthropic.APIConnectionError) as e:
        raise RuntimeError(f"LLM unavailable after retries: {type(e).__name__}") from e   # degrade gracefully
    except anthropic.BadRequestError as e:
        raise ValueError(f"Bad request, fix the code: {e.message}") from e                  # never retry
    if msg.stop_reason == "max_tokens":
        print("⚠️ answer was cut off at max_tokens")
    return {"text": text_of(msg), "stop_reason": msg.stop_reason, "first text (s)": round(first or 0, 2),
            "total (s)": round(time.perf_counter() - t0, 2), "input tokens": msg.usage.input_tokens,
            "output tokens": msg.usage.output_tokens, "cost $": round(cost(msg.usage), 6), "request id": getattr(msg, "_request_id", None)}

res = ask("Give me one tip for handling LLM API errors, in two sentences.")
print(res["text"]); {k: v for k, v in res.items() if k != "text"}

# LLM API Error Handling Tip

Implement exponential backoff with jitter when retrying failed requests—start with a short delay (e.g., 1 second) and double it on each retry, adding randomization to prevent thundering herd problems. This approach handles transient errors like rate limits and temporary outages without overwhelming the API or creating synchronized retry storms.


{'stop_reason': 'end_turn',
 'first text (s)': 0.45,
 'total (s)': 1.42,
 'input tokens': 23,
 'output tokens': 85,
 'cost $': 0.000448,
 'request id': None}

## 8. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [9]:
f = {
    "model": MODEL, "sdk": anthropic.__version__,
    "naive_seconds": round(naive_seconds, 2),
    "anatomy": anatomy.drop(columns=["request id"]).to_dict("records"),
    "cut_text": text_of(cut),
    "streaming_runs": stream_df.round(3).to_dict("records"),
    "streaming_median": stream_summary.reset_index().to_dict("records"),
    "errors": errors.to_dict("records"),
    "retries": retries.to_dict("records"),
    "timeouts": timeouts.to_dict("records"),
    "production_call": {k: v for k, v in res.items() if k not in ("text", "request id")},
}
Path("findings.json").write_text(json.dumps(f, indent=2, ensure_ascii=False, default=str))
print(json.dumps({k: f[k] for k in ("streaming_median", "errors", "retries", "timeouts", "anatomy")}, indent=1, default=str))

{
 "streaming_median": [
  {
   "mode": "non-streaming",
   "first text (s)": 3.03,
   "total (s)": 3.03,
   "output tokens": 258.0
  },
  {
   "mode": "streaming",
   "first text (s)": 0.6,
   "total (s)": 3.05,
   "output tokens": 258.0
  }
 ],
 "errors": [
  {
   "case": "wrong model name",
   "exception": "NotFoundError",
   "status": 404,
   "HTTP attempts": 1,
   "seconds": 0.23
  },
  {
   "case": "invalid request (empty messages list)",
   "exception": "BadRequestError",
   "status": 400,
   "HTTP attempts": 1,
   "seconds": 0.21
  },
  {
   "case": "bad API key",
   "exception": "AuthenticationError",
   "status": 401,
   "HTTP attempts": 1,
   "seconds": 0.14
  },
  {
   "case": "server unreachable",
   "exception": "APIConnectionError",
   "status": "-",
   "HTTP attempts": 3,
   "seconds": 1.42
  }
 ],
 "retries": [
  {
   "max_retries": 0,
   "succeeded": "23/40",
   "success %": 57,
   "HTTP attempts": 40,
   "injected failures": 17,
   "seconds": 13.7
  },
  {
   "max_re